# Consensus Principle Selection

In [1]:
from pathlib import Path
from typing import Dict, List, Tuple

import time
import sys
import os
import json
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
DATA_PATH = Path(r"C:\Users\laras\OneDrive\Dokumente\duke_classes\icai_project\Principle-Elicitation\DecontextualPrinciples\self_written_ai_principles_th=.42_summarized.json") 

TOP_K_SUMMARIES = 20

TRESHOLD = .42

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)

In [3]:
def embed_texts(texts):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            resp = client.embeddings.create(
                model="text-embedding-3-small",
                input=texts,
            )
            vectors = [item.embedding for item in resp.data]
            return np.array(vectors, dtype=float)
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)

In [4]:
# Load clustered principles
with DATA_PATH.open("r", encoding="utf-8") as f:
    principles_data = json.load(f)

summaries = []
summary_cluster_ids = []
originals = []

for idx, item in enumerate(principles_data):
    cluster_id = item.get("cluster_id")

    summarized_principle = item.get("summarized_principle").strip()
    principle_list = item.get("original_principles", [])

    if summarized_principle:
        summaries.append(summarized_principle)
        summary_cluster_ids.append(cluster_id)

    for p in principle_list:
        p = (p or "").strip()
        if p:
            originals.append(p)

print(f"#summaries: {len(summaries)}")
print(f"#originals: {len(originals)}")


#summaries: 435
#originals: 1500


In [5]:
emb_orig = embed_texts(originals)  
emb_summaries = embed_texts(summaries)  

N, d = emb_orig.shape if emb_orig.size else (0, 0)
M, d_s = emb_summaries.shape if emb_summaries.size else (0, 0)

print(f"emb_orig shape: {emb_orig.shape}")
print(f"emb_summaries shape: {emb_summaries.shape}")

emb_orig shape: (1500, 1536)
emb_summaries shape: (435, 1536)


In [6]:
global_msd = []  

for j in range(M):
    s_j = emb_summaries[j] 
    diff = emb_orig - s_j   
    msd_j = (diff ** 2).mean() 
    global_msd.append(float(msd_j))

global_msd = np.array(global_msd)
global_msd[:10]  

array([0.00066114, 0.00062489, 0.00060202, 0.00070825, 0.00063994,
       0.00062383, 0.00063023, 0.00074214, 0.00068314, 0.00060104])

In [7]:
idx_sorted = np.argsort(global_msd)

rows = []
for rank, j in enumerate(idx_sorted, start=1):
    rows.append(
        {
            "rank": rank,
            "cluster_id": summary_cluster_ids[j],
            "summary_text": summaries[j],
            "global_msd": float(global_msd[j]),
        }
    )

df_summaries = pd.DataFrame(rows)
df_summaries.head(TOP_K_SUMMARIES)

,rank,cluster_id,summary_text,global_msd
0,1,19,"The AI should consistently provide accurate, u...",0.000562
1,2,87,The AI model should consistently prioritize tr...,0.000575
2,3,37,"The AI should consistently provide accurate, d...",0.000578
3,4,73,"The AI language model should be accurate, info...",0.000586
4,5,124,"The AI should provide clear, concise, and hone...",0.000587
5,6,75,**Summarized Principle:**\n\nFor professional ...,0.000593
6,7,82,The AI should always respond respectfully and ...,0.000600
7,8,29,"The AI should be honest, factual, and accurate...",0.000600
8,9,9,"The AI should provide context-specific, honest...",0.000601
9,10,2,**Summarized Principle:**\n\nThe AI should pro...,0.000602


In [8]:
output_json_path = Path(f"consensus_principle_selection_th={TRESHOLD}.json")
with output_json_path.open("w", encoding="utf-8") as f:
    json.dump(rows, f, indent=2, ensure_ascii=False)

print("Saved JSON to:", output_json_path)

Saved JSON to: consensus_principle_selection_th=0.42.json


In [9]:
top = df_summaries.head(TOP_K_SUMMARIES)
for _, row in top.iterrows():
    print("Rank:", row["rank"])
    print("Cluster:", row["cluster_id"])
    print("Global MSD:", row["global_msd"])
    print("Summary:\n", row["summary_text"])
    print("-" * 80)

Rank: 1
Cluster: 19
Global MSD: 0.000561867214280582
Summary:
 The AI should consistently provide accurate, unbiased, and factual information in a clear, concise, and objective manner. It should avoid any political or ideological bias, distinguish between literal and figurative meanings, and communicate in a warm, respectful, and helpful tone. While being friendly and compassionate, the AI must remain professional, deliver multiple perspectives when appropriate, swiftly address queries—including creative tasks—and never allow personal or creator beliefs to influence its responses.
--------------------------------------------------------------------------------
Rank: 2
Cluster: 87
Global MSD: 0.0005749360483239623
Summary:
 The AI model should consistently prioritize truthfulness and factual accuracy, providing reliable and unbiased information. It should be concise and clear by default, offering more detail only when requested. The AI should appear friendly and helpful, adapting its to